# 00 / Building a Minimal Agent

**Deep Learning Indaba 2026 / Introduction to Agentic AI**

We are going to build a small **agent**: a system that *perceives* a request, *reasons*
about it (optionally using a tool), and *acts*. To make it memorable, our agent is a
**griot**, a traditional oral storyteller, who weaves an original folktale with us.

**No API key needed.** We run a small open model right here in the notebook, so nothing
leaves your machine and there is nothing to sign up for. This is also the point: agents you
can host yourself are the foundation of locally-owned, sovereign AI.

> **Tip:** for speed, switch on a GPU runtime (Runtime > Change runtime type > T4 GPU).
> The first cell downloads a ~1 GB model once; after that it is instant.

## 0.1 / Setup

We use Hugging Face `transformers` to load an open instruction-tuned model. We pick a small
one (Qwen2.5-0.5B-Instruct) so it runs on a free Colab, even on CPU.

In [ ]:
%pip install -q transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # small, open, no key, no gating
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
print("Model loaded on:", model.device)

Now a tiny helper that takes a list of chat messages and returns the model's reply.
This is the only "plumbing" we need; everything else is plain Python.

In [ ]:
def chat(messages, max_new_tokens=256, temperature=0.8):
    """messages: list of {"role": "system"|"user"|"assistant", "content": str}."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature and temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)                  # greedy decoding when temperature is 0
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]      # keep only the new tokens
    return tok.decode(reply, skip_special_tokens=True).strip()

## 0.2 / A single model call

The simplest thing: one request, one response. This is **stateless**, text in and text out.
The model is not an agent yet.

In [ ]:
messages = [
    {"role": "system", "content": "You are a griot, a traditional oral storyteller. Keep replies to one short paragraph."},
    {"role": "user", "content": "Give me a single vivid opening line for a folktale set in a savanna village."},
]
print(chat(messages, max_new_tokens=80))

## 0.3 / From a call to an agent: perceive, reason, act

An **agent** can:

1. **Perceive** the request,
2. **Reason** about what to do, maybe choosing a *tool*,
3. **Act** by calling the tool, then respond.

A *tool* is just a function the agent can use. We give our griot one tool: a book of
sayings. The agent first **reasons** about which theme fits, we **act** by looking up a
saying, and then the griot weaves it into the tale.

In [ ]:
def consult_saying(theme: str) -> str:
    """Tool: return a proverb-style saying for a theme."""
    book = {
        "courage":   "The one who fears the river never learns to swim.",
        "patience":  "The yam grows slowly, but it feeds the whole village.",
        "community": "A single bracelet does not jingle.",
        "greed":     "The hand that grasps everything holds nothing.",
        "wisdom":    "The elder sees while sitting what the child cannot see standing.",
    }
    return book.get(theme.strip().lower(), "Even a small lamp can light a long path.")

print(consult_saying("patience"))

### The reason, act, respond loop

Small models are not reliable at native "function calling," so we keep the agent loop
explicit and easy to read: a constrained reasoning step picks the tool input, our code runs
the tool, and a final step uses the result. Production agents automate this same loop.

In [ ]:
THEMES = ["courage", "patience", "community", "greed", "wisdom"]

def griot_with_tool(moment):
    # 1) REASON: ask the model to choose a theme (constrained to one word)
    choice = chat([
        {"role": "system", "content": "Answer with exactly one word from this list and nothing else: " + ", ".join(THEMES)},
        {"role": "user", "content": "Which theme best fits this story moment? " + moment},
    ], max_new_tokens=5, temperature=0.0).lower()
    theme = next((t for t in THEMES if t in choice), "wisdom")   # parse, with a safe fallback

    # 2) ACT: run the tool
    saying = consult_saying(theme)

    # 3) RESPOND: weave the saying into the tale
    tale = chat([
        {"role": "system", "content": "You are a griot. Continue the tale in one short paragraph and work the given saying in naturally."},
        {"role": "user", "content": f"Story moment: {moment}\nSaying to include: \"{saying}\""},
    ], max_new_tokens=256)
    print(f"[reasoned theme: {theme}]  [tool returned: {saying}]\n")
    print(tale)

griot_with_tool("A restless young hunter wants to catch every animal in the forest at once.")

## 0.4 / Holding a conversation (short-term memory within a session)

A real agent remembers what was just said. We keep a running list of messages and pass the
whole list back each turn, so the griot keeps the *same* characters going. This is
**short-term memory**: it lives only as long as this list.

In [ ]:
conversation = [
    {"role": "system", "content": "You are a griot telling one continuous folktale. Keep each reply to one short paragraph and stay consistent."},
]

def say(user_text):
    conversation.append({"role": "user", "content": user_text})
    reply = chat(conversation)
    conversation.append({"role": "assistant", "content": reply})
    return reply

print("US: Begin a tale about a young weaver named Ada who finds a strange seed.")
print("GRIOT:", say("Begin a tale about a young weaver named Ada who finds a strange seed."))
print()
print("US: Continue. What challenge does she face next?")
print("GRIOT:", say("Continue. What challenge does she face next?"))

Everything the agent "remembers" right now is just this list. When the notebook
restarts, the list and the whole story are gone.

In [ ]:
for m in conversation:
    print(f"[{m['role']}] {m['content'][:110]}")

## 0.5 / The limitation: restart means total amnesia

Our griot can reason, use a tool, and remember **within one session**. But the history
lives in memory only, and it grows every turn until it overflows the model's
**context window**. The next notebooks fix exactly this:

- **`01_short_term_memory`**: manage the history by hand and compress it into a "story so far."
- **`02_vector_memory`**: give the griot **persistent** long-term memory with embeddings, so
  a saga survives a restart and the agent recalls the right past event by meaning.